In [29]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.output_parsers import PydanticOutputParser

In [30]:
from pydantic import BaseModel, Field
from typing import Literal

In [31]:
class sentiment_review(BaseModel):
    sentiment: Literal['positive', 'negative', 'neutral'] = Field(description='Sentiment of the review')

In [32]:
parser = PydanticOutputParser(pydantic_object=sentiment_review)

In [33]:
prompt1 = PromptTemplate(
    template='Find the sentiment of the below text \n {text} \n {format_instruction}',
    input_variables=['text'],
    partial_variables={'format_instruction': parser.get_format_instructions()}
)

In [34]:
prompt2 = PromptTemplate(
    template='Write in response of the {positive} sentiment',
    input_variables=['positive']
)

In [35]:
prompt3 = PromptTemplate(
    template='Write in response of the {negative} sentiment',
    input_variables=['negative']
)

In [36]:
load_dotenv()
model = ChatOpenAI(model='gpt-4.1', temperature=0.1, max_completion_tokens=400)

In [37]:
review = """
Ah, the modern smartphone: a $1,000 rectangle specifically engineered to ensure you never have to experience the "horror" of a single original thought or a moment of eye contact ever again.

It’s truly a pinnacle of human achievement. We’ve managed to fit the entire sum of human knowledge into our pockets, only to use it primarily for arguing with strangers about movie casting and watching high-definition videos of a golden retriever eating a watermelon.

It’s also a fantastic personal assistant. It "helpfully" notifies you that you’ve spent six hours today scrolling through "productivity hacks," which is exactly the kind of irony that makes life worth living. Plus, the battery life is perfectly calibrated to hit 1% the exact moment you actually need to use the GPS to find your way out of a parking garage.

Truly, we are living in the future."""

In [38]:
initial_chain = prompt1 | model | parser

In [39]:
result = initial_chain.invoke({'text':review})

In [40]:
result

sentiment_review(sentiment='negative')

In [41]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

In [42]:
from langchain_core.output_parsers import StrOutputParser

In [43]:
parser2 = StrOutputParser()

In [44]:
prompt4 = PromptTemplate(
    template='Write in response of the {neutral} review.',
    input_variables=['neutral']
)

In [45]:
conditional_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser2),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser2),
    (lambda x:x.sentiment == 'neutral', prompt4 | model | parser2),
    RunnableLambda(lambda x: "No sentiment found")
)

In [46]:
final_chain = initial_chain | conditional_chain

In [47]:
final_result = final_chain.invoke({'text':review})

In [49]:
result

sentiment_review(sentiment='negative')

In [48]:
final_result

"I'm sorry to hear that you're feeling this way. If you'd like to share more about what's going on, I'm here to listen and help however I can."